# Series 2.7 — Model Routing

**Why AI Fails? — Engineering Lab**

---

> Enterprise AI becomes efficient by selecting the **right model** — not the biggest model every time.

**Scenario:** 1,000 synthetic AI requests across translation, SQL, code review, architecture, legal analysis, and vision — four routing strategies.

**Core lesson:** The best AI system knows **when** to use the large model.

## 1. The Problem

| Single-model baseline | Intelligent routing |
|-----------------------|---------------------|
| Every request → large general LLM | Simple tasks → small / medium models |
| Highest cost on every call | Average cost drops dramatically |
| Same latency for "translate hello" and "design architecture" | Latency matches task complexity |
| No escalation path | Confidence routing escalates only when needed |

### Why this matters in production

- Most enterprise traffic is **simple** (classification, translation, summarization)
- Routing wrong → either **overpay** (small task on large model) or **under-serve** (complex task on small model)
- Security policy may require **internal** models for confidential code

**Four strategies:**

```
single     → everything to large general LLM (baseline)
rules      → static task-type → model map
dynamic    → score models on intent, complexity, cost, latency
confidence → start cheap; escalate when confidence is low
```

## 2. What is Model Routing?

**Model routing** selects which LLM (or model tier) handles each request based on task type, complexity, cost, latency, and security policy.

### Definition

```
Model routing = request → classify → estimate complexity → policy check → pick model → respond
```

### Model pool in this lab

| Model | Best for |
|-------|----------|
| Small Language Model | Translation, classification, summarization |
| Medium Coding Model | SQL, API development, code review |
| Medium Coding (Internal) | Restricted / confidential code |
| Large Reasoning Model | Architecture, legal, complex reasoning |
| Vision Model | Image analysis |
| Large General LLM | Baseline — all tasks, highest cost |

### What model routing is NOT

| Technique | Difference |
|-----------|------------|
| **Smaller prompt** (2.1–2.6) | Routing picks the **model**; prior labs shrink **input** |
| **Load balancing** | Routing is **task-aware**, not round-robin |
| **Fallback on error** | Confidence routing escalates on **low confidence**, not API failure only |

## 3. Repository Layout

```
why-ai-fails/
├── common/
└── series-2.7/
    ├── app.py           ← CLI benchmark entry
    ├── models.py        ← Model pool (cost, latency, strengths)
    ├── requests.py      ← 1,000 synthetic AI requests
    ├── classifier.py    ← Intent + task classification
    ├── complexity.py    ← Simple / medium / complex
    ├── policy.py        ← Security + budget rules
    ├── router.py        ← Four routing strategies
    ├── evaluator.py     ← Accuracy, utilization, escalation
    ├── benchmark.py
    ├── README.md
    └── Series_2.7_Model_Routing.ipynb   ← This notebook
```

## 4. Python Files in This Lab

Every `.py` file under `series-2.7/`:

| File | What it does |
|------|--------------|
| **`app.py`** | CLI entry point. Routes 1,000 requests through four strategies, aggregates cost/latency/accuracy, supports `--request-id` inspection. |
| **`models.py`** | Enterprise model pool — `MODELS` dict with cost, latency, quality, capabilities; `get_model()`, `estimate_model_cost()`. |
| **`requests.py`** | Generates 1,000 synthetic AI requests with task type, complexity, security level, and expected model tier. |
| **`classifier.py`** | Keyword-based `detect_intent()` and `classify_task()` — no ML deps. |
| **`complexity.py`** | `estimate_complexity()` — simple / medium / complex from prompt signals. |
| **`policy.py`** | Security + budget rules — `allowed_models()`, `apply_security_policy()` restricts internal/confidential code to approved models. |
| **`router.py`** | Four routing strategies: `single`, `rules`, `dynamic`, `confidence`. `route_request()`, static `RULE_MAP`, escalation ladder. |
| **`evaluator.py`** | Routing accuracy, model utilization %, escalation rate aggregation. |
| **`prompts.py`** | `build_routing_prompt()` for live Gemini execution of routed requests. |
| **`benchmark.py`** | `run_strategy()`, side-by-side comparison and utilization report. |

## 5. The Routing Pipeline (`router.py`)

```
User Request
    ↓
Intent Detection + Task Classification
    ↓
Complexity Estimation
    ↓
Security Policy + Cost Evaluation
    ↓
Model Router (single / rules / dynamic / confidence)
    ↓
Best Model → Response
```

| Strategy | CLI | Approach |
|----------|-----|----------|
| Single | `--strategy single` | Baseline — all → large general |
| Rules | `--strategy rules` | Static `RULE_MAP` by task type |
| Dynamic | `--strategy dynamic` | Score all allowed models |
| Confidence | `--strategy confidence` | Cheap first; escalate one tier if confidence < 0.72 |

## 6. Three Layers of Routing Engineering

### Layer 1 — Classify the request

`classifier.py` detects intent and task type from the request text. Wrong classification → wrong route.

---

### Layer 2 — Apply policy and score models

Security policy restricts confidential code to internal models. Dynamic routing scores remaining models on fit, cost, and latency.

---

### Layer 3 — Measure accuracy vs cost

| Metric | What it tells you |
|--------|-------------------|
| **Routing accuracy** | Model matches expected tier |
| **Average cost** | Mean estimated cost per request |
| **Model utilization** | Traffic distribution across pool |
| **Escalation rate** | Confidence routing upgrades |

| Mode | Flag | API key? |
|------|------|----------|
| **Dry-run** | `--dry-run` | No — **$0** |
| **Live** | (none) | Yes — single request demo |

## 7. Execution Flow

```
Parse CLI (--strategy, --request-id, --requests)
    │
    └─ Load request dataset (default 1,000)
            For each request (or one via --request-id):
                classify_task() → estimate_complexity()
                apply_security_policy() → route()
                evaluate accuracy + cost
            └─ print_benchmark()
```

## 8. How to Run

From the **repo root**:

```bash
pip install -r requirements.txt
cp .env.example .env   # optional
```

| Command | What it does | API key? |
|---------|--------------|----------|
| `python series-2.7/app.py --dry-run` | All four strategies on 1k requests | No |
| `python series-2.7/app.py --strategy confidence --dry-run` | Escalation routing | No |
| `python series-2.7/app.py --request-id r0025 --dry-run` | Inspect one request | No |
| `python series-2.7/app.py --requests 100 --dry-run` | Faster dev test | No |

In [ ]:
# Live demo cell — run the dry-run benchmark ($0, no API key needed)
# Execute this cell during your presentation

import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "demo.py").exists() and (ROOT.parent / "demo.py").exists():
    ROOT = ROOT.parent

result = subprocess.run(
    [sys.executable, str(ROOT / "series-2.7/app.py"), "--dry-run"],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
print(f"\nExit code: {result.returncode}")


## 9. Key Code Snippets

### Rule map (`router.py`)

```python
RULE_MAP = {
    "translation": "small",
    "sql_generation": "medium_coding",
    "architecture_design": "large_reasoning",
    ...
}
```

### Confidence escalation

```python
CONFIDENCE_THRESHOLD = 0.72
ESCALATION_STEP = {
    "small": "medium_coding",
    "medium_coding": "large_reasoning",
    ...
}
```

## 10. Where Series 2.7 Fits

Series 2.7 **completes the cost stack**:

| Lab | Layer |
|-----|-------|
| 2.1 | Prune evidence |
| 2.2 | Cache stable prompts |
| 2.3 | Chunk & retrieve documents |
| 2.4 | Summarize conversations |
| 2.5 | Compress long-term memory |
| 2.6 | Retrieve memories at scale |
| **2.7** | **Route to the right model** |

---

## Takeaway

> **Select the right model — not the biggest model.**  
> Prune, cache, retrieve, summarize — then route what remains.

**Previous lab:** [Series 2.6 — Memory Retrieval](../series-2.6/)